# 1. Initial Setup and Parameters
- Embedding dimension d_model=8
- KV cache dimension kv_latent_dim=4
- Number of heads n_heads=2
- Head dimension dh=d_model/n_heads=4
- Input token embedding x=[0.1,-0.2,.3,-.4,.5,-.6,.7,-.8]
- Prior context 5 tokens("The next day is bright")

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
class RopelessMLA(nn.Module):
    def __init__(self,d_model,n_heads,kv_latent_dim):
        super().__init__()
        self.d_model=d_model
        self.n_heads=n_heads
        self.dh=d_model//n_heads

        # projection layer
        self.w_q=nn.Linear(d_model,d_model,bias=False)
        self.w_dkv=nn.Linear(d_model,kv_latent_dim,bias=False)
        self.w_uk=nn.Linear(kv_latent_dim,d_model,bias=False)
        self.w_uv=nn.Linear(kv_latent_dim,d_model,bias=False)
        self.w_o=nn.Linear(d_model,d_model,bias=False)

        self.ln=nn.RMSNorm(kv_latent_dim)
        self.register_buffer('absorbed_k',None) # holds w_q @ w_uk

    def forward(self,x,kv_cache=None,past_length=0):
        B,S,D=x.size()

        #compute absorbed_k once: w_q@w_uk, shape: (D,latent_dim)
        if self.absorbed_k is None:
            absorbed=torch.matmul(self.w_q.weight,self.w_uk.weight)
            self.absorbed_k=absorbed.view(self.n_heads,self.dh,-1) # (n_heads,dh,latent_dim)

        # compress x into latent kv cache
        new_c_kv=self.ln(self.w_dkv(x)) # (B,S,latent_dim)
        if kv_cache is None:
            c_kv=new_c_kv
        else:
            c_kv=torch.cat([kv_cache,new_c_kv],dim=1) # (B,S_total,latent_dim)

        S_full=c_kv.size(1) # total len S after cat cache and input

        # decompress v to full d_model and split into heads
        v_full=self.w_uv(c_kv) # (B,s_full,D)
        v=v_full.view(B,S_full,self.n_heads,self.dh).transpose(1,2) # (B,n_heads,s_full,dh)

        # use input x directly (since w_q is absorbed)
        q=x.view(B,S,self.n_heads,self.dh) # (B,S,n_heads,dh)

        # compute attention scores
        attn_scores=torch.zeros((B,self.n_heads,S,S_full),device=x.device)
        for h in range(self.n_heads):
            tmp=torch.matmul(q[:,:,h],self.absorbed_k[h])
            attn_scores[:,h]=torch.bmm(tmp,c_kv.transpose(1,2))

        # scale and apply causal mask
        attn_scores=attn_scores/(self.dh**.5)
        mask=torch.tril(torch.ones((S,S_full),device=x.device),diagonal=past_length)
        attn_scores=attn_scores.masked_fill(mask.view(1,1,S,S_full)==0,float('-inf'))

        # softmax to get attention weights
        attn_weights=F.softmax(attn_scores,dim=-1)

        # apply attention weights to each head's v separately
        out_heads=[]
        for h in range(self.n_heads):
            context_h=torch.matmul(attn_weights[:,h],v[:,h]) # (B,S,dh)
            out_heads.append(context_h)

        # concatenate all  head outputs along the feature dimension
        out=torch.cat(out_heads,dim=-1) # (B,S,D)

        return self.w_o(out), c_kv # finally output projection + updated latent cache

# Step 2. Demo for Testing

In [9]:
# =========Demo for Testing=========
def demo():
    model=RopelessMLA(d_model=512,n_heads=8,kv_latent_dim=256)
    x=torch.randn(1,5,512)
    out,cache=model(x)
    print(f'output:{out.shape},cache:{cache.shape}')

    #memory comparison
    std_size=2*2*10*512*4/1024
    latent_size=2*10*256*4/1024
    print(f"memory:standard={std_size:.1f}KB,latent={latent_size:.1f}KB,Reduction={std_size/latent_size:.1f}x")

demo()

output:torch.Size([1, 5, 512]),cache:torch.Size([1, 5, 256])
memory:standard=80.0KB,latent=20.0KB,Reduction=4.0x


Step 3: Cache testing-single new inference

In [10]:
def demo_cache_usage():
    torch.manual_seed(0)

    model=RopelessMLA(d_model=8,n_heads=8,kv_latent_dim=4)

    # step 1: initial input
    x1=torch.randn(1,5,8)
    out,cache1=model(x1)
    print('step 1: Initial input')
    print(f'output shape: {out.shape}')
    print(f'cache shape: {cache1.shape}')

    # step2: add one token
    x2=torch.randn(1,1,8)
    out2,cache2=model(x2,kv_cache=cache1,past_length=5)
    print('\nstep 2: new token')
    print(f'output shape: {out2.shape}')
    print(f'cache shape: {cache2.shape}')
demo_cache_usage()

step 1: Initial input
output shape: torch.Size([1, 5, 8])
cache shape: torch.Size([1, 5, 4])

step 2: new token
output shape: torch.Size([1, 1, 8])
cache shape: torch.Size([1, 6, 4])


step 4: cache testing multiple new inferences

In [13]:
def demo_kv_cache_growth(num_initial_tokens=50,num_new_tokens=4):
    model=RopelessMLA(d_model=8,n_heads=8,kv_latent_dim=4)
    x_initial=torch.randn(1, num_initial_tokens,8)
    out,cache=model(x_initial)
    print(f'step 0: initial cache shape: {cache.shape}')
    for i in range(1,num_new_tokens+1):
        x_new=torch.randn(1,1,8)
        out,cache=model(x_new,kv_cache=cache,past_length=cache.shape[1])
        print(f"step {i}: add one token->cache shape:{cache.shape}")
demo_kv_cache_growth()

step 0: initial cache shape: torch.Size([1, 50, 4])
step 1: add one token->cache shape:torch.Size([1, 51, 4])
step 2: add one token->cache shape:torch.Size([1, 52, 4])
step 3: add one token->cache shape:torch.Size([1, 53, 4])
step 4: add one token->cache shape:torch.Size([1, 54, 4])
